In [1]:
%matplotlib inline
import os
import sys
import glob
import numpy as np
import pandas as pd
from datetime import datetime
import xarray as xr
from multiprocessing import Pool
import multiprocessing

from read_deterministic_data import load_GFS_datasets, load_ECMWF_datasets, read_preprocessed_IVT_data
from calc_funcs import format_timedelta_to_HHMMSS
from cw3e_tools import remove_tmp_data_files
from plotter import plot_time_height_meteograms

model_name = 'GFS' ## GFS or ECMWF
print('Creating WV Flux Meteograms for {0}'.format(model_name))

## FIRST LOOP - LEAD TIME ##
if model_name == 'ECMWF':
    arr1 = np.arange(0, 72+3, 3)
    arr2 = np.arange(78, 168+6, 6)
    F_lst = np.concatenate((arr1, arr2), axis=0)
elif model_name == 'GFS':
    F_lst = np.arange(0, 168+3, 3)

fdate='2025021818' ## set this to None to get most recently downloaded data

ERROR 1: PROJ: proj_create_from_database: Open of /home/dnash/miniconda3/envs/SEAK-impacts/share/proj failed


Creating WV Flux Meteograms for GFS


In [ ]:
#################################
### CHECK TO REMOVE TMP FILES ###
#################################
print('Removing tmp intermediate data files...') 
# Specify the directory and the pattern
tmp_directory = "/data/projects/operations/wvflux_meteograms/data/tmp/"

pattern = "tmp_{0}*.nc".format(model_name)  # Delete all tmp files
remove_tmp_data_files(tmp_directory, pattern)

In [ ]:
# ### 000 - prate
# ### 003 - filter_by_keys={'stepType': 'accum', 'typeOfLevel': 'surface'} | tp or acpcp
# ### 006 - filter_by_keys={'stepType': 'accum', 'typeOfLevel': 'surface'} | tp or acpcp

# filename = '/data/projects/external_datasets/GFS/processed/{0}/{0}_F009.grb2'.format(fdate)
# ds = xr.open_dataset(filename,
#                         engine='cfgrib',backend_kwargs={"indexpath": ''}, filter_by_keys={'stepType': 'accum', 'typeOfLevel': 'surface'})
# ds

In [ ]:
# %%time
# ####################################
# ### TEST WITH A FEW NON-PARALLEL ###
# ####################################
# for i, F in enumerate(F_lst):
#     start_time = pd.Timestamp.today()
#     #######################
#     ### LOAD MODEL DATA ###
#     #######################
#     print('... Loading {0} data for {1} hour lead'.format(model_name, F))
#     if model_name == 'ECMWF':
#         s = load_ECMWF_datasets(F=F, fdate=fdate)
#         model_data= s.calc_vars()

#     elif model_name == 'GFS':
#         s = load_GFS_datasets(F=F, fdate=fdate)
#         model_data = s.calc_vars()
        
# model_data

In [ ]:
############################################
### PREPROCESS INTERMEDIATE GFS OR ECMWF ###
############################################
print('...preprocess intermediate data ...')

def multiP_preprocess_intermediate(F):
    ##############################
    ### PREP INTERMEDIATE DATA ###
    ##############################
    print('... Loading data for {0} hour lead'.format(F))
    if model_name == 'ECMWF':
        s = load_ECMWF_datasets(F=F, fdate=fdate)
        model_data = s.calc_vars()

    elif model_name == 'GFS':
        s = load_GFS_datasets(F=F, fdate=fdate)
        model_data, tmp = s.calc_vars()

if __name__ == '__main__':
        
    with Pool(processes=30) as pool:
        print("Via map with exception")
        print("\tKicking off pool via map with exception", flush=True)
        pool.map(multiP_preprocess_intermediate,F_lst)
        print("\tKicked off pool via map with exception", flush=True)
        pool.close()
        pool.join()

In [8]:
##############################
### LOAD INTERMEDIATE DATA ###
##############################
# tmp_directory = "/home/dnash/comet_data/tmp/"
tmp_directory = "/data/projects/operations/wvflux_meteograms/data/tmp/"
files = glob.glob(tmp_directory + 'tmp_{0}_*.nc'.format(model_name))
sorted_files = sorted(files)
ds = xr.open_mfdataset(sorted_files, engine='netcdf4', concat_dim="step", combine='nested')

## need to fix precipitation
if model_name == 'ECMWF':
    tp = ds.tp.fillna(0) # make sure nan at beginning is 0 to do calculation right
    tp = tp.diff(dim='step') # calculate difference between time steps
    ds = ds.drop_vars(["tp"]) # get rid of old tp (accumulated variable)
    ds = xr.merge([ds, tp]) # merge dataset with new tp
    
if model_name == 'GFS':
    ts_3hr = pd.timedelta_range(start='0 day', periods=57, freq='3H')
    ts_6hr = pd.timedelta_range(start='0 day', periods=29, freq='6H')
    tp = ds.tp ## pull out tp
    prec_3hr = tp.sel(step=ts_3hr[1::2]) ## grab only the 3hr values
    tp2 = tp.diff(dim='step') ## calculate difference in precip
    ## the values for 6hr timesteps are correct, the values for 3hr timesteps are incorrect
    prec_6hr = tp2.sel(step=ts_6hr[1:]) # grab only the 6hr values
    new_prec = prec_3hr.combine_first(prec_6hr) # combine the correct 3hr values with the correct 6hr values
    ds = ds.drop_vars(["tp"]) # get rid of old tp (accumulated variable)
    ds = xr.merge([ds, new_prec]) # merge dataset with new tp
    
ivt = read_preprocessed_IVT_data(model=model_name, F_lst=F_lst, fdate=pd.to_datetime(ds.time.values).strftime('%Y%m%d%H'))
ds = ds.assign(ivt=(['step','latitude','longitude'],ivt.ivt.values))
## load ds into memory
ds = ds.load()
ds

<xarray.Dataset>
Dimensions:                (step: 57, isobaricInhPa: 33, latitude: 44,
                            longitude: 64)
Coordinates:
  * step                   (step) timedelta64[ns] 00:00:00 ... 7 days 00:00:00
  * isobaricInhPa          (isobaricInhPa) float64 1e+03 975.0 ... 225.0 200.0
  * latitude               (latitude) float64 26.0 27.0 28.0 ... 67.0 68.0 69.0
  * longitude              (longitude) float64 186.0 187.0 188.0 ... 248.0 249.0
    time                   datetime64[ns] 2025-02-18T18:00:00
    valid_time             (step) datetime64[ns] 2025-02-18T18:00:00 ... 2025...
    atmosphereSingleLayer  float64 0.0
    surface                float64 0.0
    isothermZero           float64 0.0
Data variables:
    u                      (step, isobaricInhPa, latitude, longitude) float32 ...
    v                      (step, isobaricInhPa, latitude, longitude) float32 ...
    r                      (step, isobaricInhPa, latitude, longitude) float32 ...
    pwat                   (step, latitude, longitude) float32 29.85 ... 1.711
    t                      (step, isobaricInhPa, latitude, longitude) float32 ...
    orog                   (step, latitude, longitude) float32 0.05937 ... 130.8
    sp                     (step, latitude, longitude) float32 1.014e+05 ... ...
    gh                     (step, latitude, longitude) float32 4.235e+03 ... 0.0
    wvflux                 (step, isobaricInhPa, latitude, longitude) float64 ...
    freezing_level_pres    (step, latitude, longitude) float32 597.6 ... 1.01...
    tp                     (step, latitude, longitude) float32 nan nan ... 0.0
    ivt                    (step, latitude, longitude) float64 281.2 ... 36.91
Attributes: (12/29)
    GRIB_paramId:                             131
    GRIB_dataType:                            fc
    GRIB_numberOfPoints:                      1038240
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_name:                                U component of wind
    GRIB_shortName:                           u
    GRIB_units:                               m s**-1
    long_name:                                U component of wind
    units:                                    m s**-1
    standard_name:                            eastward_wind

In [9]:
def subset_ds_func(ds, lat, lon, duration, model_name):
    if (duration == 7) & (model_name == 'ECMWF'):
        ts = pd.timedelta_range(start='0 day', periods=29, freq='6H')
        xtick_dur = 2
    elif (duration == 7) & (model_name == 'GFS'):
        ts = pd.timedelta_range(start='0 day', periods=57, freq='3H')
        xtick_dur = 4
    else:
        ts = pd.timedelta_range(start='0 day', periods=25, freq='3H')
        xtick_dur = 2
    
    ## subset to current point and duration length
    ds = ds.sel(latitude=lat, longitude=360-lon, step=ts, method='nearest')
    
    return ds

In [13]:
lat = 48
lon = 125
dur = 3

subset_ds = subset_ds_func(ds, lat, lon, dur, model_name)
ds = subset_ds
# subset_ds
# plot_time_height_meteograms(subset_ds, 'wvflux', lat, lon, model_name, dur)

In [14]:
## find where IVT >=250 and IWV >=20 for shading
idx = (ds.ivt >=250) & (ds.pwat >=20.)
s = pd.Series(xr.where(idx, True, False))
grp = s.eq(False).cumsum()
arr = grp.loc[s.eq(True)] \
         .groupby(grp) \
         .apply(lambda x: [x.index.min(), x.index.max()])

In [15]:
arr

3       [3, 8]
16    [22, 22]
17    [24, 24]
dtype: object

In [17]:
xs = ds.valid_time.values

In [33]:
3600*6

21600

In [28]:
np.timedelta64(5400, 's')

numpy.timedelta64(5400,'s')

In [32]:
from datetime import timedelta
for i in range(len(arr)):
    if dur == 3:
        start = xs[arr.iloc[i][0]] - np.timedelta64(5400, 's')
        stop = xs[arr.iloc[i][1]] + np.timedelta64(5400, 's')
    else:
        start = xs[arr.iloc[i][0]] - np.timedelta64(21600, 's')
        stop = xs[arr.iloc[i][1]] + np.timedelta64(21600, 's')
    print(start, stop)
    AR_dur = stop - start
    print(AR_dur / np.timedelta64(1, 'h'))

2025-02-19T01:30:00.000000000 2025-02-19T19:30:00.000000000
18.0
2025-02-21T10:30:00.000000000 2025-02-21T13:30:00.000000000
3.0
2025-02-21T16:30:00.000000000 2025-02-21T19:30:00.000000000
3.0


In [ ]:
import numpy as np
lat_lst = np.arange(26, 51, 1)
lon_lst = np.arange(111, 128, 1)
var_lst = ['r', 'wvflux']
dur_lst = [7, 3]

arglst_USWEST = []
for i, x in enumerate(lon_lst):
    for j, y in enumerate(lat_lst):
        for k, varname in enumerate(var_lst):
            for l, dur in enumerate(dur_lst):
                subset_ds = subset_ds_func(ds, y, x, dur, model_name)
                arglst_USWEST.append((subset_ds, x,y, varname, dur))

lat_lst = np.arange(51, 70, 1)
lon_lst = np.arange(130, 175, 1)               
arglst_AK = []
for i, x in enumerate(lon_lst):
    for j, y in enumerate(lat_lst):
        for k, varname in enumerate(var_lst):
            for l, dur in enumerate(dur_lst):
                subset_ds = subset_ds_func(ds, y, x, dur, model_name)
                arglst_AK.append((subset_ds, x,y, varname, dur))
                
final_arglst = arglst_USWEST + arglst_AK

for i, argval in enumerate(final_arglst[:2]):
    tmp, lon, lat, varname, dur = argval
    print(lon, lat, varname, dur)

In [ ]:
# %%time
# for argval in final_arglst[0:30]:
#     lon, lat, varname, dur = argval
#     plot_time_height_meteograms(ds, varname, lat, 360-lon, model_name, dur)

In [ ]:
del ds

In [ ]:
print('...create plots ...')

def multiP_create_time_height_meteograms(argval):    
    ds, lon, lat, varname, dur = argval
    print('plotting {3}-day {2} at {0}N, {1}W'.format(lat, lon, varname, dur), flush=True)
    plot_time_height_meteograms(ds, varname, lat, 360-lon, model_name, dur)
    print('finished plotting {3}-day {2} at {0}N, {1}W'.format(lat, lon, varname, dur), flush=True)

if __name__ == '__main__':
        
    with Pool(processes=16) as pool:
        print("Via map with exception")
        print("\tKicking off pool via map with exception", flush=True)
        pool.map(multiP_create_time_height_meteograms,final_arglst)
        print("\tKicked off pool via map with exception", flush=True)
        pool.close()
        pool.join()